---
last_verified: 2026-09-27
tool_version: n/a
---

# Choosing between Rego and YAML rules in Terrascan

> Reference notebook. Terrascan rule definitions are a YAML envelope around a Rego
> body, and a separate YAML surface configures which rules run and how findings are
> routed. This notebook separates the two surfaces and gives a decision procedure for
> placing a candidate check on the right one.

## Purpose

A candidate finding arrives and has to be routed somewhere. Three outcomes are
possible, and they cost very different amounts of work:

1. A built-in rule already covers the check, so only scan configuration changes.
2. The check is not covered but is a single-attribute predicate, so a minimal Rego
   body in a YAML policy document is enough.
3. The check needs reasoning across resources, iteration, or absence inference, so it
   needs a real Rego body.

Confusing these three is the common failure. Writing Rego for outcome 1 duplicates a
rule the tool already maintains. Writing YAML for outcomes 2 and 3 produces a policy
document whose envelope loads but whose body never fires. This notebook makes the
choice explicit and repeatable.

## When to use

- A new check has been proposed and someone has to decide where it belongs.
- A policy document is loading but reporting nothing, and the question is whether the
  defect is in the envelope or the body.
- Severity triage is inconsistent, which usually means metadata is being used to carry
  logic it was never meant to carry.

## Prerequisites

- A Terrascan binary on `PATH`, for the scan command assembled in Step 4.
- `jq`, for the severity gate in Step 4.
- The policy document loaded in Step 1, which lives in the repository at
  `terrascan/configs/tried-custom-s3-rule.yaml`.

## Step 1 — Separate the two YAML surfaces

Terrascan uses YAML in two unrelated places, and conflating them is the root of most
policy-authoring confusion.

**The policy envelope** describes one rule: an `apiVersion`, a `kind`, a `metadata`
block carrying name, severity, category, cloud provider, and resource type, and a
`spec` block holding the Rego source. This is the `terrascan/configs/` surface.

**The scan configuration** describes the run rather than any single rule: which rule
IDs are skipped, how output is produced, and what the process exits with. This is what
a `--config-path` argument points at when it is *not* pointing at a policy directory.

The first cell loads the envelope surface and splits it into the two halves that later
cells score separately.

## Step 2 — Decide whether the envelope is enough

The envelope can carry a rule's identity and nothing else. Name, severity, category,
cloud provider, and resource type describe a check; they do not perform one. So the
first question is not "Rego or YAML" but "how much logic does this rule need".

The cell below classifies each policy body by the language features it depends on, and
separates the ones that cost real reasoning from the ones that are presentational:

- `some` — the body iterates over a collection rather than looking at one resource.
- `not` — the body infers an absence, which has no declarative equivalent.
- a binding to a name other than the resource and the message — the body derives a
  value, which means it is following data across resources.
- `sprintf` — the body composes a message. Presentational, and present in almost every
  rule worth writing.

A body carrying only the presentational features is a single-resource attribute
comparison. It still needs Rego to run, but it is the smallest body that can be correct,
and it is where most new checks should land.

## Step 3 — Place the check

Given the classification, the placement follows mechanically. Outcome 1 needs no
policy document at all. Outcome 2 needs a policy document with the smallest body that
expresses the predicate. Outcome 3 needs a body that exercises iteration, binding, or
absence inference.

The function below is the whole decision. It returns the surface and the reason, so
the reason can be reviewed in the pull request that adds the rule.

## Step 4 — Wire the chosen surface into a scan

Both surfaces are loaded at scan time, and they are loaded by different arguments. A
policy directory is passed to the custom-policy argument so the Rego bodies are
compiled alongside the built-in rules; the severity gate then consumes the JSON
result. The assembled command is written out so it can be pasted into CI unchanged.

## Verify

The checks below confirm the claims this notebook makes about the artifacts it
references. They are assertions over files on disk, not over Terrascan's behaviour, so
they stay valid on a machine that has the binary and on one that does not.

## Common errors

- **Envelope-only policy.** A document with `metadata` and an empty `spec.rego` loads
  without error and reports nothing. The rule is identified but never evaluated.
- **Logic in the metadata.** Encoding a check as a resource type or a category string
  produces a rule that is named as covering something it does not check.
- **Duplicate of a built-in rule.** A Rego body that restates a maintained built-in
  rule diverges silently the first time that rule is updated.
- **Unranked severity.** A `severity` value outside `critical`, `high`, `medium`,
  `low` cannot be ordered by the gate, so the finding is reported and never fails the
  build.
- **Rego that needs a different input.** A body reasoning about relationships the
  resource document does not carry will not fire, however correct the logic is.

## References

In-repository material referenced by this notebook:

- `terrascan/configs/tried-custom-s3-rule.yaml` — the policy envelope and Rego bodies
  parsed in Steps 1 and 2.
- `terrascan/scripts/policy-as-code-workflow.sh` — loads a policy directory, emits JSON,
  and applies the severity ordering used in Step 4.
- `terrascan/docs/terrascan-vs-checkov-terraform-iac-scanning.md` — where custom Rego
  policy sits relative to a second IaC scanner.
- `terrascan/notes/0000-primer-terrascan.md` — first-contact orientation for the
  scan model and the Rego dependency.

In [ ]:
import re
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    """Walk up from the notebook's working directory until the kit layout appears."""
    for candidate in [start, *start.parents]:
        if (candidate / "terrascan" / "configs").is_dir():
            return candidate
    raise FileNotFoundError("could not locate a directory containing terrascan/configs")


REPO_ROOT = find_repo_root(Path.cwd().resolve())
POLICY_PATH = REPO_ROOT / "terrascan" / "configs" / "tried-custom-s3-rule.yaml"
print("repo root :", REPO_ROOT)
print("policy    :", POLICY_PATH.relative_to(REPO_ROOT))
print("exists    :", POLICY_PATH.is_file())

In [ ]:
def split_policy_documents(text: str) -> list[dict]:
    """Split a policy file into documents and split each into metadata vs Rego body.

    Deliberately dependency-free: the point of the notebook is the envelope/body
    split, not YAML parsing. Documents are separated on the ``---`` marker and the
    Rego body is the indented literal block under ``spec.rego``.
    """
    documents = []
    for raw in re.split(r"^---$", text, flags=re.MULTILINE):
        if "kind: policy" not in raw:
            continue
        name = re.search(r"^\s+name:\s*(\S+)", raw, re.MULTILINE)
        severity = re.search(r"^\s+severity:\s*(\S+)", raw, re.MULTILINE)
        resource_type = re.search(r"^\s+resourceType:\s*(\S+)", raw, re.MULTILINE)
        body_lines = []
        capture = False
        for line in raw.splitlines():
            if re.match(r"^\s+rego:\s*$", line):
                capture = True
                continue
            if capture:
                stripped = line.strip()
                if not stripped:
                    continue
                # the block-scalar header itself: ``rego:`` then ``- |``
                if re.match(r"^(\s*)?(- ?)?[|>][-+]?$", line):
                    continue
                if not line.startswith("      "):
                    capture = False
                    continue
                body_lines.append(stripped)
        documents.append(
            {
                "name": name.group(1) if name else "<unnamed>",
                "severity": severity.group(1) if severity else "<unset>",
                "resource_type": resource_type.group(1) if resource_type else "<unset>",
                "rego": "\n".join(body_lines).strip(),
            }
        )
    return documents


POLICY_TEXT = POLICY_PATH.read_text(encoding="utf-8")
POLICIES = split_policy_documents(POLICY_TEXT)

for policy in POLICIES:
    print(f"{policy['name']}  severity={policy['severity']}  "
          f"resourceType={policy['resource_type']}  rego_lines={len(policy['rego'].splitlines())}")

In [ ]:
PRESENTATIONAL = {"sprintf(": "composes a dynamic message"}
# Names a body may bind without following data: the resource under test and the
# message it emits. Anything else is a derived value.
STANDARD_BINDINGS = {"resource", "msg"}


def classify_body(rego: str) -> dict:
    """Report which language features a Rego body depends on.

    Returns a mapping of feature key to purpose. The key ``derived`` stands for a
    binding to a name outside STANDARD_BINDINGS, which is the only kind of ``:=``
    that indicates dataflow rather than bookkeeping.
    """
    features = {
        key: purpose for key, purpose in PRESENTATIONAL.items() if key in rego
    }
    if "some" in rego:
        features["some"] = "iterates over a collection"
    if "not " in rego:
        features["not "] = "infers an absence"
    derived = sorted(
        {name for name in re.findall(r"(\w+)\s*:=", rego)} - STANDARD_BINDINGS
    )
    if derived:
        features["derived"] = f"binds derived value(s) {derived}"
    return features


REASONING_FEATURES = {"some", "not ", "derived"}


def needs_reasoning(features: dict) -> bool:
    """True when the body does more than compare one resource's attributes."""
    return bool(REASONING_FEATURES & set(features))


for policy in POLICIES:
    features = classify_body(policy["rego"])
    print(f"{policy['name']}")
    print(f"  features  : {features or 'single-resource attribute comparisons only'}")
    print(f"  reasoning : {needs_reasoning(features)}")
    print(f"  envelope  : identity + triage routing, carried by "
          f"severity={policy['severity']}")

In [ ]:
def place_check(*, covered_by_builtin: bool, features: dict) -> tuple[str, str]:
    """Return (surface, reason) for a candidate check.

    surface is one of:
      "config-only"  - a built-in rule covers it; adjust scan configuration
      "minimal-rego" - one resource, attribute predicates, no reasoning
      "full-rego"    - needs iteration, binding, or absence inference
    """
    if covered_by_builtin:
        return "config-only", "a built-in rule already reports this check"
    present = REASONING_FEATURES & set(features)
    if present:
        return "full-rego", f"body reasons over {sorted(present)}"
    if features:
        return "minimal-rego", f"presentational features only: {sorted(features)}"
    return "minimal-rego", "single-resource attribute comparison"


CANDIDATES = [
    # Already reported by a maintained built-in rule.
    ("open storage bucket in an approved exception list", True, {}),
    # One resource, attribute comparison, message interpolation.
    ("bucket public-access block set to false", False,
     classify_body(POLICIES[0]["rego"])),
    # Needs every bucket in the module considered together.
    ("no bucket in the module allows public ACLs", False,
     {"some": "iterates over a collection",
      "derived": "binds derived value(s) ['candidate']"}),
]

for label, builtin, features in CANDIDATES:
    surface, reason = place_check(covered_by_builtin=builtin, features=features)
    print(f"{label:52s} -> {surface:13s} ({reason})")

In [ ]:
IAC_DIR = "terraform/"
POLICY_DIR = "terrascan/configs"
RESULT_JSON = "terrascan-results/scan.json"

scan_command = (
    f"terrascan scan -d {IAC_DIR} --config-path {POLICY_DIR} "
    f"-o json > {RESULT_JSON}"
)
print(scan_command)
print()

# Severity ordering used by the gate. Ordering is fixed; a lower rank is more severe.
SEVERITY_RANK = {"critical": 0, "high": 1, "medium": 2, "low": 3}

CANDIDATE_SEVERITIES = [policy["severity"] for policy in POLICIES]
print("severities declared in the envelope:", CANDIDATE_SEVERITIES)
print("ranks:", {s: SEVERITY_RANK.get(s, "unknown") for s in CANDIDATE_SEVERITIES})
print()
print("Gating on 'high' selects the first policy only:",
      [p["name"] for p in POLICIES if SEVERITY_RANK[p["severity"]] <= SEVERITY_RANK["high"]])

In [ ]:
COMPANION_FILES = [
    "terrascan/configs/tried-custom-s3-rule.yaml",
    "terrascan/scripts/policy-as-code-workflow.sh",
    "terrascan/docs/terrascan-vs-checkov-terraform-iac-scanning.md",
    "terrascan/notes/0000-primer-terrascan.md",
]

for relative in COMPANION_FILES:
    target = REPO_ROOT / relative
    assert target.is_file(), f"missing referenced file: {relative}"
    print(f"ok  {relative}")

assert len(POLICIES) == 2, f"expected 2 policy documents, parsed {len(POLICIES)}"
for policy in POLICIES:
    assert policy["rego"].startswith("package terrascan"),         f"{policy['name']}: body is not a Rego module"
    assert policy["severity"] in SEVERITY_RANK,         f"{policy['name']}: severity {policy['severity']} is outside the gate's ordering"
    print(f"ok  {policy['name']} declares a parsable envelope and body")

print()
print("all verification checks passed")